In [1]:
import os
import time
import json
import pickle
import pandas as pd
import numpy as np

from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

/opt/conda/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
2025-11-14 13:39:14.003893: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
env_path = '../../conf/local/.env'
load_dotenv(env_path)
huggingface_acess_token = os.getenv("huggingface_jp")

In [1]:
!nvidia-smi

Wed Nov 26 10:16:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     Off |   00000000:A3:00.0 Off |                    0 |
|  0%   39C    P8             35W /  300W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
torch.cuda.is_available()

True

In [6]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Load dataset

In [7]:
test_data_path = '../../data/01_processed/pan21-authorship-verification-test.jsonl'

test_data_file_size = os.path.getsize(test_data_path)
test_data = []

with open(test_data_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

Loading data: 100%|██████████| 873M/873M [00:08<00:00, 101MB/s]  


Successfully loaded 19999 items.


Load the data into the pandas dataframe

In [8]:
test_data_df = pd.DataFrame(test_data)
test_data_df.drop(columns=['fandoms'], inplace=True)
print(test_data_df.head())

                                     id  \
0  c28e8b03-c02a-5184-b58a-12dd28b8ca74   
1  b9326101-6352-56dd-9d1b-1f41466897b7   
2  e2ac4453-bf54-53f2-bf68-6caae6aacded   
3  a5e9a289-0999-5764-b597-dc1bf8c21ede   
4  cb4054b1-d422-58d6-a137-dcfc70100df6   

                                                pair  
0  [talk because they hadn"t been exposed to comm...  
1  [Zazuki nodded his head and got to his feet, k...  
2  ["Oh we did lots of special things. On Christm...  
3  ["Hey now, at least Shido brings home some mon...  
4  [It was a mere five minutes" walk from third y...  


Load the testing set labels

In [9]:
test_data_truth_path = '../../data/01_processed/pan21-authorship-verification-test-truth.jsonl'

test_data_truth_file_size = os.path.getsize(test_data_truth_path)
test_data_truth = []

with open(test_data_truth_path, 'r') as f:
    with tqdm(total=test_data_truth_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data_truth.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data_truth)} items.")

Loading data: 100%|██████████| 1.91M/1.91M [00:00<00:00, 5.21MB/s]


Successfully loaded 19999 items.


Load the data into pandas dataframe

In [10]:
test_data_truth_df = pd.DataFrame(test_data_truth)
test_data_truth_df.drop(columns=['authors'], inplace=True)
print(test_data_truth_df.head())

                                     id   same
0  c28e8b03-c02a-5184-b58a-12dd28b8ca74   True
1  b9326101-6352-56dd-9d1b-1f41466897b7   True
2  e2ac4453-bf54-53f2-bf68-6caae6aacded  False
3  a5e9a289-0999-5764-b597-dc1bf8c21ede   True
4  cb4054b1-d422-58d6-a137-dcfc70100df6   True


Combine the dataframes for evaluation

In [11]:
test_data_full_df = pd.merge(test_data_df, test_data_truth_df, on='id')
test_data_full_reduced_df =  test_data_full_df.head(100)
print(test_data_full_df.head())

                                     id  \
0  c28e8b03-c02a-5184-b58a-12dd28b8ca74   
1  b9326101-6352-56dd-9d1b-1f41466897b7   
2  e2ac4453-bf54-53f2-bf68-6caae6aacded   
3  a5e9a289-0999-5764-b597-dc1bf8c21ede   
4  cb4054b1-d422-58d6-a137-dcfc70100df6   

                                                pair   same  
0  [talk because they hadn"t been exposed to comm...   True  
1  [Zazuki nodded his head and got to his feet, k...   True  
2  ["Oh we did lots of special things. On Christm...  False  
3  ["Hey now, at least Shido brings home some mon...   True  
4  [It was a mere five minutes" walk from third y...   True  


# Set up model - llama-3.1-8B

Log in to huggingface

In [12]:
login(huggingface_acess_token)

Load the llama-3.1-8B model




In [13]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

llama3 = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    model_kwargs={"device_map": "auto"},
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Device set to use cuda:0


In [14]:
llama3.generation_config.pad_token_id = tokenizer.pad_token_id

### Set up prompts

from [https://github.com/baixianghuang/authorship-llm/blob/main/code/github-verify-llama2-70b.ipynb](https://github.com/baixianghuang/authorship-llm/blob/main/code/github-verify-llama2-70b.ipynb)

In [15]:
prompt_message = """
Verify if the input texts were written by the same author.
Do not consider topic differences. Focus on grammatical styles. 
Analyze the writing styles of the input texts, disregarding the differences in topic and content. 
Focus on linguistic features such as phrasal verbs, modal verbs, punctuation, rare words, affixes, quantities, 
humor, sarcasm, typographical errors, and misspellings. 
"""

In [16]:
response_message = """
Respond with a JSON object including two key elements:
{
  "analysis": Reasoning behind your answer.
  "answer":  A boolean (True/False) answer.
}
"""

# Evaluate model

from [https://github.com/baixianghuang/authorship-llm/blob/main/code/github-verify-llama2-70b.ipynb](https://github.com/baixianghuang/authorship-llm/blob/main/code/github-verify-llama2-70b.ipynb)

Evaluation function

In [17]:
def evaluate_results(y_test, y_pred, average='binary'):
    accuracy = round(accuracy_score(y_test, y_pred)*100, 2)
    precision = round(precision_score(y_test, y_pred, average=average)*100, 2)
    recall = round(recall_score(y_test, y_pred, average=average)*100, 2)
    f1 = round(f1_score(y_test, y_pred, average=average)*100, 2)
    return accuracy, precision, recall, f1

Evaluate model function

In [18]:
def evaluate_model(test_data_df, llm_model, prompt_message, response_message):
    start_time = time.time()
    verification_results = []

    for i in tqdm(test_data_df.index, desc="Processing rows"):

        resulting_df_row = {}
        resulting_df_row['id']  = test_data_df.loc[i, 'id']
        
        text1, text2 = test_data_df.loc[i, 'pair']

        resulting_df_row['text1'] = text1.strip()
        resulting_df_row['text2'] = text2.strip()
        resulting_df_row['actual_result'] = test_data_df.loc[i, 'same']

        llm_prompt = f"""
        <system>
        You are an authorship-verification model. {response_message}
        </system>
        
        <user>
        {prompt_message}
        
        Here are the texts:
        ```Text 1: {text1}, \n\nText 2: {text2}```
        </user>"""
    
        raw_response = llama3(
            llm_prompt,
            do_sample=False,
            temperature=0.01,
            top_p=1,
            max_new_tokens=500
        )
        raw_response_str = raw_response[0]["generated_text"]
        last_close_brace_index = raw_response_str.rfind('}')
        last_open_brace_index = raw_response_str.rfind('{', 0, last_close_brace_index)
        json_string = raw_response_str[last_open_brace_index:last_close_brace_index + 1]
        try:
            response_json = json.loads(json_string)
            resulting_df_row['reasoning'] = response_json['analysis']
            resulting_df_row['prediction'] = response_json['answer']

        except Exception as e:
            resulting_df_row['reasoning'] = json_string
            if 'True' in json_string or 'true' in json_string:
                resulting_df_row['prediction'] = True
            elif 'False' in json_string or 'false' in json_string:
                resulting_df_row['prediction'] = False
            else:
                resulting_df_row['prediction'] = None
        
        verification_results.append(resulting_df_row)

    result_df = pd.DataFrame(verification_results)
    print("--- Execution Time: %s seconds ---" % round(time.time() - start_time, 2))
    return result_df

Evaluate llama

In [19]:
result_df = evaluate_model(test_data_full_reduced_df, llama3, prompt_message, response_message)

Processing rows:   0%|          | 0/100 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Processing rows:   1%|          | 1/100 [00:51<1:25:29, 51.82s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Processing rows:   2%|▏         | 2/100 [01:40<1:21:26, 49.86s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Processing rows:   3%|▎         | 3/100 [02:28<1:19:41, 49.30s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more 

--- Execution Time: 4918.01 seconds ---


Export the results file

In [20]:
date_str = datetime.now().strftime("%Y%m%d_%H%M")
print(date_str)
file_path = f"../../results/{date_str}_JP-third_prototype_Llama-3.1-8B-Instruct_results.csv"
result_df.to_csv(file_path, index=False)

20251114_1501


In [21]:
imported_result_df = pd.read_csv(file_path)
cleaned_df = imported_result_df.dropna()

In [22]:
y_test = cleaned_df['actual_result'].to_list()
y_pred = cleaned_df['prediction'].to_list()

In [23]:
accuracy, precision, recall, f1 = evaluate_results(y_test, y_pred)
print(f"The results of evaluation are: \n Accuracy: {accuracy} \n Precision: {precision} \n Recall: {recall} \n F1 score: {f1}")

The results of evaluation are: 
 Accuracy: 51.0 
 Precision: 50.54 
 Recall: 94.0 
 F1 score: 65.73
